# 04 — From-scratch DPO on UltraFeedback (Task 4, helpfulness track)

Accelerator: **GPU T4 x2**. Policy = SFT `best` (fp32, `cuda:0`); frozen reference = SFT `best` (fp16, `cuda:1`).
Guide configuration in `configs/dpo_uf.yaml`: β = 0.1, fixed LR 1e-6, batch 1 × accumulation 32, max length 512 / prompt 256.

**Before running:** Add Input → the saved output of notebook 02 (contains `outputs/sft/best`).
Run with **Save Version → Save & Run All (Commit)**.

In [ ]:
# Clone (or update) the project repo into /kaggle/working. Notebook settings: Internet ON.
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {cmd}")

In [ ]:
# Manual section 5.2: keep Kaggle's PyTorch; do not reinstall it.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
# Manual section 7: confirm two T4 GPUs and print package versions.
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
# Locate artifacts from earlier notebooks attached under /kaggle/input (Add Input -> your notebook output).
import glob, os
def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Attach the notebook output that contains {pattern}"
    print(pattern, "->", hits[0])
    return hits[0]

In [ ]:
SFT_BEST = os.path.dirname(find_input("sft/best/checkpoint_info.json"))
!cat {SFT_BEST}/checkpoint_info.json

In [ ]:
# Re-create the same seed-42 subset (deterministic; compare with data/manifests/uf_*_ids.json).
sh("python scripts/prepare_data.py --out_dir data/processed --which uf")
!wc -l data/processed/uf_*.jsonl

In [ ]:
# Optional short pilot to check memory and speed:
# !python scripts/train_dpo.py --config configs/dpo_uf.yaml --set sft_checkpoint={SFT_BEST} max_steps=5 eval_every=5 max_val_examples=64 output_dir=outputs/dpo_uf_pilot

In [ ]:
sh(f"python scripts/train_dpo.py --config configs/dpo_uf.yaml --set sft_checkpoint={SFT_BEST}")

In [ ]:
!cat outputs/dpo_uf/summary.json
!python scripts/plot_metrics.py --run dpo_uf=outputs/dpo_uf --out_dir outputs/plots
from IPython.display import Image, display
display(Image("outputs/plots/dpo_uf_metrics.png"))

In [ ]:
# Manual section 5.5: archive the run for easy attaching/downloading, then use "Save Version".
!cd outputs && zip -q -r -0 /kaggle/working/dpo_uf_run_outputs.zip dpo_uf
!ls -lh /kaggle/working/*.zip
!du -sh outputs/dpo_uf